In [ ]:
from pathlib import Path

import pandas as pd


### 1. Paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "labeled_metrics.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "metrics_labeling.csv"
)



### 2. Main preparation function

In [ ]:
def prepare_dataset_for_split() -> None:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(
            f"Input file not found: {INPUT_PATH}"
        )

    print(f"Loading dataset: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)

    initial_rows = len(df)

    required_columns = [
        "valid_5min_horizon",
        "slowdown_in_5min",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    # Convert the validation column safely to boolean.
    # This works if the CSV contains True/False strings
    # or actual boolean values.
    if df["valid_5min_horizon"].dtype != bool:
        df["valid_5min_horizon"] = (
            df["valid_5min_horizon"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            })
        )

    if df["valid_5min_horizon"].isna().any():
        invalid_values = (
            df.loc[
                df["valid_5min_horizon"].isna(),
                "valid_5min_horizon",
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "The valid_5min_horizon column contains "
            f"unrecognized values: {invalid_values}"
        )

    # Keep only observations with a complete future horizon.
    df = df[
        df["valid_5min_horizon"] == True
    ].copy()

    valid_rows = len(df)
    removed_invalid_rows = initial_rows - valid_rows

    # Remove rows where the target is still missing.
    missing_target_before = (
        df["slowdown_in_5min"].isna().sum()
    )

    df = df.dropna(
        subset=["slowdown_in_5min"]
    ).copy()

    removed_missing_target = (
        valid_rows - len(df)
    )

    # Convert target to integer 0/1.
    df["slowdown_in_5min"] = (
        df["slowdown_in_5min"].astype(int)
    )

    unexpected_targets = set(
        df["slowdown_in_5min"].unique()
    ) - {0, 1}

    if unexpected_targets:
        raise ValueError(
            "The target contains unexpected values: "
            f"{unexpected_targets}"
        )

    # Technical/helper columns that should not be used
    # as model features.
    columns_to_drop = [
        "valid_5min_horizon",
        "valid_10min_horizon",
        "slowdown_in_10min",
    ]

    df = df.drop(
        columns=columns_to_drop,
        errors="ignore",
    )

    # Optional: sort the dataset before the split.
    sort_columns = [
        column
        for column in [
            "machine_id",
            "run_id",
            "segment_id",
            "timestamp",
        ]
        if column in df.columns
    ]

    if sort_columns:
        df = (
            df.sort_values(sort_columns)
            .reset_index(drop=True)
        )
    else:
        df = df.reset_index(drop=True)

    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df.to_csv(
        OUTPUT_PATH,
        index=False,
    )

    print("\nDataset preparation completed.")
    print(f"Initial rows: {initial_rows}")
    print(
        "Rows removed because "
        f"valid_5min_horizon=False: "
        f"{removed_invalid_rows}"
    )
    print(
        "Rows with missing target before removal: "
        f"{missing_target_before}"
    )
    print(
        "Rows removed because target was missing: "
        f"{removed_missing_target}"
    )
    print(f"Final rows: {len(df)}")
    print(
        "Positive labels:",
        int(df["slowdown_in_5min"].sum()),
    )
    print(
        "Negative labels:",
        int(
            (df["slowdown_in_5min"] == 0).sum()
        ),
    )
    print(f"Output saved to: {OUTPUT_PATH}")


In [ ]:
if __name__ == "__main__":
    prepare_dataset_for_split()